# Automation interface

If you are an experimental chemist you may be able to create some code which processes and stored your data.

One example of this in the cell below:

In [ ]:
import csv
from pathlib import Path

class Compound:
    def __init__(self, name: str, mf: str):
        self.name = name
        self.mf = mf

class Sample:
    def __init__(self, location: str, mfs: list[str]):
        self.location = location
        self.masses = mfs

def generate_masslynx_metadata(expt_name: str) -> list[dict[str, str | int]]:
    """
    Generates initial metadata lines for a MassLynx sample sequence, including a blank and QC sample.

    Args:
        expt_name (str): The experiment name to use in file naming.

    Returns:
        list[dict[str, str | int]]: A list containing two row dictionaries, one for the blank sample and one 
            for the QC sample, formatted for MassLynx input.
    """
    # Generate a blank sample with 0-uL injection at the start, to stabilize/check for ghost peaks
    # Location must exist but doesn't matter where
    blank_line = {'Index': 1, 'FILE_NAME': f'{expt_name}-BLANK-OPT', 'SAMPLE_LOCATION':'1:01',
                'INJ_VOL': 0, # Remember to update the injection volume to 0 for the blank line
                'MASS_A':'', 'MASS_B':'', 'MASS_C':''
                }
    # Merge blank line dict into fixed parameters dict, blank line values taking precedence
    blank_line = {**fixed_params, **blank_line}

    # Generate a QC sample of test mix to run at the start of the sequence, for data normalization
    qc_line = {'Index': 2, 'FILE_NAME': f'{expt_name}-QC-OPT', 'SAMPLE_LOCATION': TEST_MIX_LOCATION, 
               'MASS_A':'', 'MASS_B':'', 'MASS_C':'',
                }
    qc_line = {**fixed_params, **qc_line} # Merge QC line dict into fixed parameters dict

    return [blank_line, qc_line]

def generate_sample_list(products: list[Compound], 
                         plate_location: str, 
                         wellplate_size: int = 48
                         ) -> list[Sample]:
    """
    Generates a list of Sample objects for a wellplate, each containing the same set of product masses 
    for reaction optimization.

    Args:
        products (list[Compound]): List of Compound objects representing the products in each well.
        plate_location (str): Plate identifier or location prefix for each well.
        wellplate_size (int, optional): Number of wells in the plate. Defaults to 48.

    Returns:
        list[Sample]: List of Sample objects, one for each well in the plate.
    """
    sample_list = []
    # This is rxn optimization so here, products are the same in each well
    masses = [product.mf for product in products] # MassLynx takes either MF or MW; MF is preferred
    for i in range(wellplate_size):
        location = f'{plate_location}:{i+1:02d}' # Well location as 2-digit string with leading zeros
        sample = Sample(location, masses)
        sample_list.append(sample)

    return sample_list

def generate_masslynx_lines(expt: str, sample_list: list[Sample]) -> list[dict[str, str | int]]:
    """
    Generates a list of dictionaries representing rows for a MassLynx input file, including blank, QC, and sample lines.

    Args:
        expt (str): The experiment name to use in file naming.
        sample_list (list[Sample]): List of Sample objects representing the samples to include.

    Returns:
        list[dict[str, str | int]]: List of dictionaries, each representing a row for the MassLynx input file.
    """
    # Instaniate a list to hold dictionaries of line data, which will be written to the MassLynx input file as rows
    lines = []
    # Generate a blank injection and QC injection of test mix at the start
    lines = generate_masslynx_metadata(expt) # This will return a list of dicts, which is added to the lines list

    # Add a line for each sample
    for n, sample in enumerate(sample_list):
        line_dict = {**fixed_params, # Start with the fixed parameters for all lines, then add row-specific params
                     'Index': n + 2, # Add 2 to n to account for the blank and QC lines at the start of the sequence
                     'FILE_NAME': f'{expt}-{sample.location}-OPT', # Generate file name based on experiment name and sample location
                     'SAMPLE_LOCATION': sample.location, 
                     # Add masses; MassLynx requires exactly 3 so they will be padded or truncated as needed
                     'MASS_A': sample.masses[0] if len(sample.masses) > 0 else '', # Mass A, if it exists
                     'MASS_B': sample.masses[1] if len(sample.masses) > 1 else '', # Mass B, if it exists
                     'MASS_C': sample.masses[2] if len(sample.masses) > 2 else '' # Mass C, if it exists
                     }
        lines.append(line_dict)

    return lines

def write_masslynx_file(expt_name: str, sample_list: list[Sample]) -> None:
    """
    Writes a MassLynx-compatible CSV file for a given experiment and sample list.

    Args:
        expt_name (str): The experiment name, used as the output CSV filename prefix.
        sample_list (list[Sample]): List of Sample objects to include in the file.

    Side Effects:
        Creates or overwrites a CSV file named '<expt_name>.csv' in the current working directory.

    Note: 
        - The CSV file will be overwritten if it already exists. 
    """
    with open(Path(f'{expt_name}.csv'), 'w', newline = '', encoding = 'ANSI', errors = 'replace') as fout:
        writer = csv.writer(fout)
        writer.writerow(headers) # Write tuple of headers to first row

        for line_dict in generate_masslynx_lines(expt_name, sample_list):
            line = [line_dict[header] for header in headers] # Ensure values are in the same order as headers
            # Encode each value to ANSI, replacing unencodable characters, and decode back to string
            encoded_line = [str(value).encode('ANSI', errors='replace').decode('ANSI') for value in line]
            writer.writerow(encoded_line)

# User-defined experimental metadata
expt_name = 'G26-000057'
plate_location = '1' # Must be string; the instrument does not require integers
plate_size = 48 # 48-well plate
# This is a reaction optimization screen, so the potential products are the same in every well
products = [Compound('2-pyridone', 'C5H5NO'), Compound('2-pyridone-d5', 'C5H0D5NO')]
anal_method = '5-min acid'
MS_method = 'MS scan 5 min'
proc_method = 'Autopurify_5min'

# Define global MassLynx parameters
TEST_MIX_LOCATION = '0:02' # Fixed location of test mix for QC injections
# Note: the contents of headers is specific to your MassLynx setup
headers = ('Index', 'FILE_NAME', 'SAMPLE_LOCATION', 'INJ_VOL', 'PREP_INJ_VOL',
            'INLET_FILE', 'MS_FILE', 'MASS_A', 'MASS_B', 'MASS_C', 'PROCESS_PARAMS', 'PROCESS'
            )
# Set some fixed parameters that will be the same for all lines, to avoid repetition
fixed_params = {'INJ_VOL':1, 'PREP_INJ_VOL':1600,
                'INLET_FILE':anal_method, 'MS_FILE':MS_method, 
                'PROCESS_PARAMS':proc_method, 'PROCESS':'AutoPurify'
                }

if __name__ == '__main__':
    sample_list = generate_sample_list(products, plate_location, plate_size)
    write_masslynx_file(expt_name, sample_list)